# Heed: train a wake word on your own voice

Train a tiny (sub-250 KB) wake word tuned to your own microphone and voice. You
add a handful of real recordings of yourself, and training mixes in hundreds of
synthetic speakers so the model still generalizes. At the end you test it on your
own voice right here, then download a model that runs in the browser, on a phone,
or in Python.

```
  your clips  -----+
                   +--> + hundreds of TTS voices --> augment (noise / room / EQ) --> train --> wake.onnx + wake.json
  hard negatives --+
```

If you just want a generic model with zero recording, use the simpler
`heed_train_colab.ipynb` instead.

Switch the runtime to GPU (Runtime, Change runtime type, T4 GPU). It works on CPU
too, just slower. Project: https://github.com/AndreiBulzan/heed-wakeword


## 1. Install

In [ ]:
# A GPU runtime (Runtime > Change runtime type > T4 GPU) makes TTS and training much faster.
!pip install -q "heed-wakeword[tts,export]"
!heed --version

## 2. Browser-recording helper

Defines `record_to_wav()`, used by the recording and testing cells below. It needs
microphone permission in the Colab output frame. If recording does not work in
your browser, use the upload cells instead, they do the same job.

In [ ]:
from IPython.display import display, Javascript
from google.colab import output
from base64 import b64decode
from pathlib import Path
import subprocess, uuid

_RECORD_JS = '''
async function recordAudio(seconds) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const rec = new MediaRecorder(stream);
  const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  const stopped = new Promise(res => { rec.onstop = res; });
  rec.start();
  await new Promise(r => setTimeout(r, seconds * 1000));
  rec.stop();
  await stopped;
  stream.getTracks().forEach(t => t.stop());
  const buf = await (new Blob(chunks)).arrayBuffer();
  const bytes = new Uint8Array(buf);
  let bin = '';
  for (let i = 0; i < bytes.length; i++) bin += String.fromCharCode(bytes[i]);
  return btoa(bin);
}
'''

def record_to_wav(out_path, seconds=2):
    display(Javascript(_RECORD_JS))
    b64 = output.eval_js(f'recordAudio({seconds})')
    tmp = f'/tmp/{uuid.uuid4().hex}.webm'
    Path(tmp).write_bytes(b64decode(b64))
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['ffmpeg', '-y', '-i', tmp, '-ar', '16000', '-ac', '1', str(out_path)],
                   check=True, capture_output=True)
    return out_path

print('record_to_wav() ready')

## 3. Pick your phrase and settings

In [ ]:
PHRASE = "hey jasper"  #@param {type:"string"}
N_TTS_POSITIVES = 250  #@param {type:"integer"}
EPOCHS = 35  #@param {type:"integer"}

import re
PROJECT = re.sub(r'[^a-z0-9]+', '_', PHRASE.lower()).strip('_') or 'wake'
print('phrase:', repr(PHRASE), '| project:', PROJECT)

## 4. Create the project and download the voice

In [ ]:
!heed init {PROJECT} --phrase "{PHRASE}"
!heed download-tts

## 5. Add your positives

Give it 10 to 15 clips of you saying the phrase (8 is the minimum). Either upload
existing clips (next cell) or record in the browser (the cell after). You can do both.

Recording quality is the single biggest factor in whether your wake word works,
more than any setting below. A few minutes of varied recording beats tuning later:

- Vary distance and angle. A few up close, a few at arm's length, one or two across the room.
- Vary how you say it. Normal, quiet, a little fast, a little slow. Say it like you actually would, not robotically.
- Use a real room, not a sound booth. A little natural background (a fan, distant traffic) helps; avoid dead silence and avoid loud noise.
- Say only the phrase. Start the clip, say it once, stop. Trim long silences.
- More beats perfect. 12 quick natural takes generalize better than 8 careful identical ones.

The next section adds hard negatives, and training mixes in hundreds of synthetic
speakers on top of your clips.


In [ ]:
# Upload clips (wav, mp3, m4a all fine; they are converted to 16 kHz mono).
from google.colab import files
from pathlib import Path
import subprocess

proj = Path(PROJECT)
(proj / 'positive').mkdir(parents=True, exist_ok=True)
print(f'Upload clips of you saying "{PHRASE}":')
uploaded = files.upload()
for i, (fn, data) in enumerate(uploaded.items()):
    tmp = f'/tmp/up_{i}_{Path(fn).name}'
    Path(tmp).write_bytes(data)
    out = proj / 'positive' / f'me_{i:03d}.wav'
    subprocess.run(['ffmpeg', '-y', '-i', tmp, '-ar', '16000', '-ac', '1', str(out)], check=True, capture_output=True)
print('positives now:', len(list((proj / 'positive').glob('*.wav'))))

In [ ]:
# Or record in the browser. Run this cell; for each prompt, press Enter then speak.
# The first run may error while you grant the mic permission, just run the cell again.
proj = Path(PROJECT)
N_RECORDINGS = 8  #@param {type:"integer"}
for i in range(N_RECORDINGS):
    input(f'Press Enter, then say "{PHRASE}" ({i + 1}/{N_RECORDINGS}, about 2s)... ')
    record_to_wav(proj / 'positive' / f'rec_{i:03d}.wav', seconds=2)
    print(f'  saved {i + 1}/{N_RECORDINGS}')
print('positives now:', len(list((proj / 'positive').glob('*.wav'))))

## 6. Synthesize negatives

Hard negatives (phonetic neighbors and common phrases) so the model learns the
boundary. Training adds many more on top of these.

In [ ]:
from heed.tts import synthesize_phrase, phonetic_neighbor_distractors
from heed.audio import save_wav

proj = Path(PROJECT)
(proj / 'negative').mkdir(parents=True, exist_ok=True)
distractors = phonetic_neighbor_distractors(PHRASE, max_neighbors=12)
k = 0
for text in distractors:
    for clip in synthesize_phrase(text, 2, seed=100 + k):
        save_wav(proj / 'negative' / f'neg_{k:03d}.wav', clip)
        k += 1
print(f'seed negatives: {k}  (distractors: {distractors[:6]} ...)')

## 7. Train

Uses your real positives plus `N_TTS_POSITIVES` synthetic speakers and a pool of
hard negatives. With your own voice in the mix, training also EQ-matches the
synthetic audio toward your microphone. The printed held-out evaluation is the
honest cross-speaker signal.

In [ ]:
!heed train {PROJECT} --epochs {EPOCHS} --tts-pos {N_TTS_POSITIVES}

## 7b. How good is it? (score separation)

This plots how the trained model scores your project's clips: positives (you plus
synthetic speakers saying the phrase) should pile up to the right of the threshold,
negatives to the left. Clean separation means a healthy model.

These are the training clips, so the picture is optimistic by design. The honest
cross-speaker number is the held-out evaluation printed by the training cell above.


In [ ]:
import torch, matplotlib.pyplot as plt
from heed.trainer import load_model
from heed.audio import load_dir_clips, log_mel
from heed.eval import evaluate_dirs, fmt_report

model, payload = load_model(f"{PROJECT}/model.pt")
thr = float(payload["threshold"])

@torch.no_grad()
def _scores(d):
    clips = load_dir_clips(d)
    if not clips:
        return torch.zeros(0)
    mels = torch.cat([log_mel(c) for c in clips], dim=0)
    return torch.sigmoid(model(mels)).flatten().cpu()

pos = _scores(f"{PROJECT}/positive").numpy()
neg = _scores(f"{PROJECT}/negative").numpy()

print(fmt_report(evaluate_dirs(f"{PROJECT}/model.pt", f"{PROJECT}/positive", f"{PROJECT}/negative")))

plt.figure(figsize=(7, 3.2))
plt.hist(neg, bins=20, range=(0, 1), alpha=0.75, label=f"negatives (n={len(neg)})", color="#d9534f")
plt.hist(pos, bins=20, range=(0, 1), alpha=0.75, label=f"positives (n={len(pos)})", color="#5cb85c")
plt.axvline(thr, color="k", ls="--", lw=1.5, label=f"threshold {thr:.2f}")
plt.xlabel("model score"); plt.ylabel("clips")
plt.title(f'"{PHRASE}"  -  score separation'); plt.legend(); plt.tight_layout(); plt.show()
print("Healthy: green (positives) sits right of the dashed line, red (negatives) to the left.")


## 8. Test it on your own voice

Record a clip and score it against the trained model. Run the cell again to retry.
A score above the threshold printed by training means it triggered.

In [ ]:
# The first run may error while you grant the mic permission, just run the cell again.
proj = Path(PROJECT)
test_wav = proj / 'my_test.wav'
input(f'Press Enter, then say "{PHRASE}" to test... ')
record_to_wav(test_wav, seconds=2)
!heed test {PROJECT} "{test_wav}"

## 9. Export and download

In [ ]:
!heed export {PROJECT}
import shutil
from google.colab import files
archive = shutil.make_archive(f'{PROJECT}_model', 'zip', f'{PROJECT}/export')
files.download(archive)

## What you got

A sub-250 KB model plus its `wake.json` contract, tuned to your voice but robust
across speakers. Run it in the browser (`examples/inference_browser/`), on a phone
(`examples/inference_react_native/assets/`), or in Python. See the export README in
the zip and the project docs for the deployment snippets.